# Imports

In [9]:
# Import modules
import tensorflow as tf

import numpy as np

from keras.applications.resnet50 import ResNet50
from keras.layers import Flatten, Dense, Dropout
from keras.models import Sequential, save_model
from keras.optimizers import RMSprop

from tqdm import tqdm

In [2]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent.parent))

from preprocessing.audio_processor import preprocess
from config import AUDIO_DIR, AUDIO_CONFIG

🖥️  Using device: cuda


# Data Preprocessing

In [3]:
# Load data parameters
tf.random.set_seed(42)

IMG_HEIGHT = AUDIO_CONFIG["spectrogram_height"]
IMG_WIDTH = AUDIO_CONFIG["spectrogram_width"]
CHANNELS = 3

INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)

EPOCHS = 5
BATCH_SIZE = 32

In [4]:
audio_data = {
    'training': {
        'spectrograms': [],
        'labels': []
    },
    'testing': {
        'spectrograms': [],
        'labels': []
    }
}

# Track skipped files
skipped_files = []

for split in ['training', 'testing']:
    for label in ['real', 'fake']:
        audio_files = list((AUDIO_DIR / split / label).glob('*.wav')) + list((AUDIO_DIR / split / label).glob('*.mp3'))
        print(f"\nProcessing {label} audio files in {split} set:")
        
        for audio in tqdm(audio_files, desc=f"{split}/{label}"):
            try:
                spectrogram, _ = preprocess(audio)
                
                audio_data[split]['spectrograms'].append(spectrogram)
                audio_data[split]['labels'].append(0 if label=='real' else 1)
            except Exception as e:
                # Skip corrupted or unreadable files
                skipped_files.append((str(audio), str(e)))
                continue

# Report skipped files
if skipped_files:
    print(f"\nSkipped {len(skipped_files)} corrupted/unreadable files:")
    for file_path, error in skipped_files:
        print(f"  - {file_path}")
else:
    print(f"\nAll files processed successfully!")


Processing real audio files in training set:


training/real:   0%|          | 0/4000 [00:00<?, ?it/s]

training/real: 100%|██████████| 4000/4000 [06:01<00:00, 11.06it/s]



Processing fake audio files in training set:


training/fake:  66%|██████▌   | 2648/4000 [01:46<00:26, 51.73it/s]d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\!S9\Explainability AI\Project\XAI_Final_Project\preprocessing\audio_processor.py:30: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=None)
d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\.env_A5\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
training/fake: 100%|██████████| 4000/4000 [02:40<00:00, 24.95it/s]



Processing real audio files in testing set:


testing/real: 100%|██████████| 1000/1000 [00:52<00:00, 18.96it/s]



Processing fake audio files in testing set:


testing/fake: 100%|██████████| 1000/1000 [00:34<00:00, 29.15it/s]



Skipped 1 corrupted/unreadable files:
  - d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\!S9\Explainability AI\Project\XAI_Final_Project\data\audio\training\fake\file13424.mp3


In [7]:
# Convert lists to NumPy arrays and create TensorFlow datasets
train_tensors = np.array(audio_data['training']['spectrograms'])
train_labels = np.array(audio_data['training']['labels'])

test_tensors = np.array(audio_data['testing']['spectrograms'])
test_labels = np.array(audio_data['testing']['labels'])

print(f"Train shape: {train_tensors.shape}, Labels: {train_labels.shape}")
print(f"Test shape: {test_tensors.shape}, Labels: {test_labels.shape}")

train_ds = tf.data.Dataset.from_tensor_slices((train_tensors, train_labels)).shuffle(1000).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((test_tensors, test_labels)).shuffle(100).batch(BATCH_SIZE)

print(f"\nTrain batches: {len(train_ds)}")
print(f"Test batches: {len(test_ds)}")

Train shape: (7999, 224, 224, 3), Labels: (7999,)
Test shape: (2000, 224, 224, 3), Labels: (2000,)

Train batches: 250
Test batches: 63


In [8]:
import gc

del audio_data, train_tensors, train_labels, test_tensors, test_labels

gc.collect()

2492

# Model

In [10]:
resnet = ResNet50(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE)

# Do not train the model, leverage pretrained layers
for layer in resnet.layers:
    layer.trainable = False

In [11]:
# Create transfer learning model
model_resnet = Sequential()

# Add pretrained model as it is
model_resnet.add(resnet)
# Flatten last layer of resnet
model_resnet.add(Flatten())
# Add layers of our own
model_resnet.add(Dense(512, activation='relu', input_dim=INPUT_SHAPE))
model_resnet.add(Dropout(0.3))
model_resnet.add(Dense(512, activation='relu'))
model_resnet.add(Dropout(0.3))
model_resnet.add(Dense(1, activation='sigmoid'))

model_resnet.compile(loss='binary_crossentropy', 
                     optimizer=RMSprop(learning_rate = 2e-5), 
                     metrics=['accuracy'])

model_resnet.summary()

d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\.env_A5\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    51,380,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 75,231,617 (286.99 MB)

 Trainable params: 51,643,905 (197.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [12]:
history = model_resnet.fit(
    train_ds,
    validation_data = test_ds,
    epochs = EPOCHS
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 638s 2s/step - accuracy: 0.9735 - loss: 0.0737 - val_accuracy: 0.8385 - val_loss: 0.5522
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 410s 2s/step - accuracy: 0.9890 - loss: 0.0291 - val_accuracy: 0.8955 - val_loss: 0.3862
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 407s 2s/step - accuracy: 0.9924 - loss: 0.0229 - val_accuracy: 0.9215 - val_loss: 0.3816
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.9942 - loss: 0.0184 - val_accuracy: 0.9785 - val_loss: 0.0887
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 406s 2s/step - accuracy: 0.9956 - loss: 0.0134 - val_accuracy: 0.9570 - val_loss: 0.1695


In [13]:
save_model(model_resnet, filepath="./resnet50.h5")